
# Air Quality Data Ingestion and Cleaning

## Project Overview

This notebook implements the data ingestion and preprocessing stage of the British Columbia Air Quality Analysis project.

The project uses hourly air quality observations collected from monitoring stations across British Columbia. The original data is provided in CSV format and contains measurements of air pollutants and Air Quality Health Index (AQHI) values.

## Objectives

The main objectives of this notebook are:

1. Load the original air quality CSV files using Apache Spark.
2. Inspect the structure, schema, and quality of the source data.
3. Standardize column names and missing-value representations.
4. Identify missing values, duplicate records, and negative pollutant measurements.
5. Create data-quality flags to identify potentially invalid readings.
6. Separate monitoring station observations from regional AQHI observations.
7. Validate and save the cleaned datasets in Parquet format.

## Technologies Used

- Python
- Apache Spark (PySpark)
- Pandas
- NumPy
- Parquet

## Expected Outputs

At the end of this notebook, two cleaned datasets will be produced:

- Monitoring Station Data
- Regional AQHI Data

These datasets will be used in the subsequent exploratory data analysis, machine learning, and Spark performance experiments.


In [ ]:
!java -version

[0.018s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.018s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


## 1. Environment Setup and Library Imports

This section imports the Python libraries and PySpark components required for data ingestion, data cleaning, data validation, and Parquet file processing.

In [ ]:
import os
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
# Initialize Apache Spark

spark = SparkSession.builder \
    .appName("BC Air Quality Data Ingestion and Cleaning") \
    .getOrCreate()

print("Spark session initialized successfully.")
print("Spark version:", spark.version)

In [ ]:

from pathlib import Path

# The repository root will be the current working directory
PROJECT_DIR = Path.cwd()

# Input data directory
RAW_DIR = PROJECT_DIR / "01_Raw_Data"

# Output directories
PROCESSED_DIR = PROJECT_DIR / "02_Processed_Data"
RESULTS_DIR = PROJECT_DIR / "03_Analysis_Results"
PLOTS_DIR = PROJECT_DIR / "04_Visualizations"
MODEL_DIR = PROJECT_DIR / "05_Model_Results"

# Create output directories if they do not exist
for folder in [
    PROCESSED_DIR,
    RESULTS_DIR,
    PLOTS_DIR,
    MODEL_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# Display project directory information
print("Project root:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Raw data directory exists:", RAW_DIR.exists())
print("Processed data directory:", PROCESSED_DIR)
print("Results directory:", RESULTS_DIR)


Project directories created successfully!
Raw data folder: /content/drive/MyDrive/Big_Data_Analytics/Air_Quality_Project/01_Raw_Data
Raw data folder exists: True



## 2. Data Acquisition and File Inventory

The project uses publicly available hourly air quality observations from the Government of British Columbia.

The original datasets are stored as CSV files and organized within the project's raw data directory.

Before ingestion, the available CSV files are identified and inventoried. This allows us to verify the source files, examine their sizes, and understand the volume of data being processed.

The original source files are preserved without modification.


In [ ]:

from pathlib import Path

# Locate all CSV files in the raw data directory
csv_files = sorted(RAW_DIR.rglob("*.csv"))

# Verify that CSV files are available
if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found in: {RAW_DIR}"
    )

print("Total CSV files found:", len(csv_files))

# Display the available source files
for file in csv_files:
    print(file.name)


Total CSV files found: 16
CH4.csv
CO.csv
H2S.csv
HF.csv
NMHC.csv
NO.csv
NO2.csv
NOx.csv
O3.csv
PM10.csv
PM25.csv
SNOW.csv
SO2.csv
THC.csv
TRS.csv
aqhi.csv


### CSV File Size Analysis

In [ ]:

import builtins

# Collect file names and sizes
file_inventory = []

for file in csv_files:
    file_size = file.stat().st_size / (1024 * 1024)

    file_inventory.append({
        "File Name": file.name,
        "File Size (MB)": builtins.round(file_size, 2)
    })

# Convert inventory into a Pandas DataFrame
file_inventory_df = pd.DataFrame(file_inventory)

display(file_inventory_df)

# Calculate total raw data size
total_size = file_inventory_df["File Size (MB)"].sum()

print(
    "Total raw data size:",
    builtins.round(total_size, 2),
    "MB"
)


,File Name,File Size (MB)
0,CH4.csv,0.14
1,CO.csv,1.19
2,H2S.csv,0.21
3,HF.csv,0.14
4,NMHC.csv,0.14
5,NO.csv,3.68
6,NO2.csv,3.72
7,NOx.csv,3.65
8,O3.csv,3.39
9,PM10.csv,2.18


Total file size: 33.01 MB



## 3. Data Ingestion Using Apache Spark

In this stage, the original CSV files are loaded into individual Apache Spark DataFrames using PySpark.

Each CSV file is loaded separately because the source files contain different pollutant measurements and may have different schemas.

The ingestion process uses:

- The first row as column headers.
- Automatic schema inference to identify column data types.
- PERMISSIVE mode to handle malformed records without immediately stopping the ingestion process.

Each DataFrame is stored in a Python dictionary using its original filename as the key.

The original CSV files are not modified during ingestion.


In [ ]:

# Dictionary to store individual Spark DataFrames
dataframes = {}

# Load each CSV file into a Spark DataFrame
for file in csv_files:

    df = spark.read \
        .option("header", True) \
        .option("inferSchema", True) \
        .option("mode", "PERMISSIVE") \
        .csv(str(file))

    # Store DataFrame using the original filename
    dataframes[file.name] = df

    print(f"Successfully loaded: {file.name}")

# Display ingestion summary
print("\nTotal CSV files loaded:", len(dataframes))


Successfully loaded: CH4.csv
Successfully loaded: CO.csv
Successfully loaded: H2S.csv
Successfully loaded: HF.csv
Successfully loaded: NMHC.csv
Successfully loaded: NO.csv
Successfully loaded: NO2.csv
Successfully loaded: NOx.csv
Successfully loaded: O3.csv
Successfully loaded: PM10.csv
Successfully loaded: PM25.csv
Successfully loaded: SNOW.csv
Successfully loaded: SO2.csv
Successfully loaded: THC.csv
Successfully loaded: TRS.csv
Successfully loaded: aqhi.csv

Total DataFrames loaded: 16


In [ ]:
# Display a sample of the PM2.5 dataset
if "PM25.csv" in dataframes:

    print("Sample records from PM2.5 dataset:")
    dataframes["PM25.csv"].show(5, truncate=False)

else:
    print("PM25.csv was not found in the ingested datasets.")

+-------------------+--------------------+---------+--------------+----------+-----+---------+-------+--------+---------+
|           DATE_PST|        STATION_NAME|RAW_VALUE|REPORTED_VALUE|INSTRUMENT|UNITS|PARAMETER| EMS_ID|LATITUDE|LONGITUDE|
+-------------------+--------------------+---------+--------------+----------+-----+---------+-------+--------+---------+
|2026-11-05 12:00:00|Abbotsford A Colu...|  5.71668|           5.7| PM25_T640|ug/m3|     PM25|E289309| 49.0215|-122.3265|
|2026-11-05 11:00:00|Abbotsford A Colu...| 6.152633|           6.2| PM25_T640|ug/m3|     PM25|E289309| 49.0215|-122.3265|
|2026-11-05 10:00:00|Abbotsford A Colu...| 6.404513|           6.4| PM25_T640|ug/m3|     PM25|E289309| 49.0215|-122.3265|
|2026-11-05 09:00:00|Abbotsford A Colu...| 6.744485|           6.7| PM25_T640|ug/m3|     PM25|E289309| 49.0215|-122.3265|
|2026-11-05 08:00:00|Abbotsford A Colu...| 7.413463|           7.4| PM25_T640|ug/m3|     PM25|E289309| 49.0215|-122.3265|
|2026-11-05 07:00:00|Abb


## 4. Initial Data Exploration and Quality Assessment

Before cleaning the datasets, an initial inspection is performed to understand their structure, volume, coverage, and data quality.

This section examines:

- Dataset schemas and column data types.
- Sample records from each CSV file.
- Number of observations in each file.
- Available date ranges.
- Monitoring stations represented in the datasets.
- Missing values.
- Duplicate station and timestamp observations.
- Descriptive statistics for numerical columns.

These checks help identify inconsistencies and potential data-quality issues before preprocessing.


In [ ]:
for filename, df in dataframes.items():

    print("\n" + "=" * 60)
    print("FILE:", filename)
    print("=" * 60)

    df.printSchema()


FILE: CH4.csv
root
 |-- DATE_PST: timestamp (nullable = true)
 |-- STATION_NAME: string (nullable = true)
 |-- RAW_VALUE: double (nullable = true)
 |-- REPORTED_VALUE: double (nullable = true)
 |-- INSTRUMENT: string (nullable = true)
 |-- UNITS: string (nullable = true)
 |-- PARAMETER: string (nullable = true)
 |-- EMS_ID: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)


FILE: CO.csv
root
 |-- DATE_PST: timestamp (nullable = true)
 |-- STATION_NAME: string (nullable = true)
 |-- RAW_VALUE: double (nullable = true)
 |-- REPORTED_VALUE: double (nullable = true)
 |-- INSTRUMENT: string (nullable = true)
 |-- UNITS: string (nullable = true)
 |-- PARAMETER: string (nullable = true)
 |-- EMS_ID: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)


FILE: H2S.csv
root
 |-- DATE_PST: timestamp (nullable = true)
 |-- STATION_NAME: string (nullable = true)
 |-- RAW_VALUE: double (nu

### 4.1 Sample Records

In [ ]:
for filename, df in dataframes.items():

    print("\nFILE:", filename)

    df.show(5, truncate=False)


FILE: CH4.csv
+-------------------+-------------------------+---------+--------------+-----------+-----+---------+-------+----------+----------+
|DATE_PST           |STATION_NAME             |RAW_VALUE|REPORTED_VALUE|INSTRUMENT |UNITS|PARAMETER|EMS_ID |LATITUDE  |LONGITUDE |
+-------------------+-------------------------+---------+--------------+-----------+-----+---------+-------+----------+----------+
|2026-11-05 12:00:00|Farmington Community Hall|2.008568 |2.01          |UNSPECIFIED|ppm  |CH4      |E295892|55.9132326|-120.53157|
|2026-11-05 11:00:00|Farmington Community Hall|2.0059221|2.01          |UNSPECIFIED|ppm  |CH4      |E295892|55.9132326|-120.53157|
|2026-11-05 10:00:00|Farmington Community Hall|2.00246  |2.0           |UNSPECIFIED|ppm  |CH4      |E295892|55.9132326|-120.53157|
|2026-11-05 09:00:00|Farmington Community Hall|2.002439 |2.0           |UNSPECIFIED|ppm  |CH4      |E295892|55.9132326|-120.53157|
|2026-11-05 08:00:00|Farmington Community Hall|2.0251009|2.03       

### 4.2 Record Counts

In [ ]:
row_counts = []

for filename, df in dataframes.items():

    row_counts.append({
        "File Name": filename,
        "Number of Rows": df.count()
    })

row_count_df = pd.DataFrame(row_counts)

display(row_count_df)

print(
    "Total rows across all files:",
    row_count_df["Number of Rows"].sum()
)

,File Name,Number of Rows
0,CH4.csv,1466
1,CO.csv,13194
2,H2S.csv,2199
3,HF.csv,1466
4,NMHC.csv,1466
5,NO.csv,38849
6,NO2.csv,38849
7,NOx.csv,38116
8,O3.csv,35917
9,PM10.csv,21990


Total rows across all files: 306032


### 4.3 Date Range Analysis

In [ ]:
for filename, df in dataframes.items():

    print("\nFILE:", filename)

    if "DATE_PST" in df.columns:

        df.select(
            min("DATE_PST").alias("Start_Date"),
            max("DATE_PST").alias("End_Date")
        ).show(truncate=False)

    else:
        print("DATE_PST column not found.")


FILE: CH4.csv
+-------------------+-------------------+
|Start_Date         |End_Date           |
+-------------------+-------------------+
|2026-10-06 00:00:00|2026-11-05 12:00:00|
+-------------------+-------------------+


FILE: CO.csv
+-------------------+-------------------+
|Start_Date         |End_Date           |
+-------------------+-------------------+
|2026-10-06 00:00:00|2026-11-05 12:00:00|
+-------------------+-------------------+


FILE: H2S.csv
+-------------------+-------------------+
|Start_Date         |End_Date           |
+-------------------+-------------------+
|2026-10-06 00:00:00|2026-11-05 12:00:00|
+-------------------+-------------------+


FILE: HF.csv
+-------------------+-------------------+
|Start_Date         |End_Date           |
+-------------------+-------------------+
|2026-10-06 00:00:00|2026-11-05 12:00:00|
+-------------------+-------------------+


FILE: NMHC.csv
+-------------------+-------------------+
|Start_Date         |End_Date           

### 4.4 Monitoring Station Coverage

In [ ]:
for filename, df in dataframes.items():

    print("\nFILE:", filename)

    if "STATION_NAME" in df.columns:

        stations = df.select(
            "STATION_NAME"
        ).distinct()

        print("Number of stations:", stations.count())

        stations.show(30, truncate=False)

    else:
        print("STATION_NAME column not found.")


FILE: CH4.csv
Number of stations: 2
+-------------------------+
|STATION_NAME             |
+-------------------------+
|Kitimat Riverlodge       |
|Farmington Community Hall|
+-------------------------+


FILE: CO.csv
Number of stations: 18
+----------------------------------+
|STATION_NAME                      |
+----------------------------------+
|Tsawwassen                        |
|Maple Ridge Golden Ears School    |
|Kelowna KLO Road                  |
|Port Moody Rocky Point Park       |
|Richmond South                    |
|Coquitlam Douglas College         |
|Chilliwack Airport                |
|Horseshoe Bay                     |
|Vancouver Robson Square           |
|Vancouver International Airport #2|
|Burnaby South                     |
|Vancouver Clark Drive             |
|Victoria Topaz                    |
|Surrey East                       |
|North Vancouver Mahon Park        |
|Abbotsford A Columbia Street      |
|Hope Airport                      |
|Langley Central 

### 4.5 Missing Value Assessment

In [ ]:
for filename, df in dataframes.items():

    print("\nFILE:", filename)

    missing_values = df.select([
        count(
            when(
                col(c).isNull() |
                (trim(col(c).cast("string")) == ""),
                1
            )
        ).alias(c)

        for c in df.columns
    ])

    missing_values.show(truncate=False)


FILE: CH4.csv
+--------+------------+---------+--------------+----------+-----+---------+------+--------+---------+
|DATE_PST|STATION_NAME|RAW_VALUE|REPORTED_VALUE|INSTRUMENT|UNITS|PARAMETER|EMS_ID|LATITUDE|LONGITUDE|
+--------+------------+---------+--------------+----------+-----+---------+------+--------+---------+
|0       |0           |598      |598           |0         |0    |0        |0     |0       |0        |
+--------+------------+---------+--------------+----------+-----+---------+------+--------+---------+


FILE: CO.csv
+--------+------------+---------+--------------+----------+-----+---------+------+--------+---------+
|DATE_PST|STATION_NAME|RAW_VALUE|REPORTED_VALUE|INSTRUMENT|UNITS|PARAMETER|EMS_ID|LATITUDE|LONGITUDE|
+--------+------------+---------+--------------+----------+-----+---------+------+--------+---------+
|0       |0           |3190     |3190          |0         |0    |0        |0     |0       |0        |
+--------+------------+---------+--------------+----

### 4.6 Duplicate Observation Assessment

In [ ]:
for filename, df in dataframes.items():

    print("\nFILE:", filename)

    if (
        "DATE_PST" in df.columns
        and "STATION_NAME" in df.columns
    ):

        duplicates = df.groupBy(
            "DATE_PST",
            "STATION_NAME"
        ).count().filter(
            col("count") > 1
        )

        print(
            "Duplicate station-timestamp groups:",
            duplicates.count()
        )

        duplicates.show(10, truncate=False)


FILE: CH4.csv
Duplicate station-timestamp groups: 0
+--------+------------+-----+
|DATE_PST|STATION_NAME|count|
+--------+------------+-----+
+--------+------------+-----+


FILE: CO.csv
Duplicate station-timestamp groups: 0
+--------+------------+-----+
|DATE_PST|STATION_NAME|count|
+--------+------------+-----+
+--------+------------+-----+


FILE: H2S.csv
Duplicate station-timestamp groups: 0
+--------+------------+-----+
|DATE_PST|STATION_NAME|count|
+--------+------------+-----+
+--------+------------+-----+


FILE: HF.csv
Duplicate station-timestamp groups: 0
+--------+------------+-----+
|DATE_PST|STATION_NAME|count|
+--------+------------+-----+
+--------+------------+-----+


FILE: NMHC.csv
Duplicate station-timestamp groups: 0
+--------+------------+-----+
|DATE_PST|STATION_NAME|count|
+--------+------------+-----+
+--------+------------+-----+


FILE: NO.csv
Duplicate station-timestamp groups: 0
+--------+------------+-----+
|DATE_PST|STATION_NAME|count|
+--------+---------

### 4.7 Numerical Descriptive Statistics

In [ ]:
for filename, df in dataframes.items():

    print("\nFILE:", filename)

    numeric_columns = [
        field.name
        for field in df.schema.fields
        if isinstance(
            field.dataType,
            (IntegerType, LongType, FloatType, DoubleType)
        )
    ]

    if numeric_columns:
        df.select(numeric_columns).describe().show()


FILE: CH4.csv
+-------+-------------------+-------------------+------------------+-------------------+
|summary|          RAW_VALUE|     REPORTED_VALUE|          LATITUDE|          LONGITUDE|
+-------+-------------------+-------------------+------------------+-------------------+
|  count|                868|                868|              1466|               1466|
|   mean|  2.070249132488483| 2.0704228110598986| 54.98356130000036|-124.60150300000268|
| stddev|0.06460108804149507|0.06455658201782058|0.9299885398317788|  4.071321818672035|
|    min|           1.668223|               1.67|          54.05389|        -128.671436|
|    max|          2.5765021|               2.58|        55.9132326|         -120.53157|
+-------+-------------------+-------------------+------------------+-------------------+


FILE: CO.csv
+-------+-------------------+-------------------+------------------+-------------------+
|summary|          RAW_VALUE|     REPORTED_VALUE|          LATITUDE|          LO

## 5. Data Cleaning and Standardization

In [ ]:
import re
from pyspark.sql import functions as F

# Select the AQHI dataset explicitly
if "aqhi.csv" not in dataframes:
    raise FileNotFoundError("aqhi.csv was not found.")

source_df = dataframes["aqhi.csv"]

# Function to standardize column names
def clean_column_name(name):
    name = name.strip().lower()
    name = re.sub(r'[^a-zA-Z0-9]+', '_', name)
    return name.strip('_')

# Rename columns consistently
df_clean = source_df

for old_name in df_clean.columns:
    new_name = clean_column_name(old_name)

    if old_name != new_name:
        df_clean = df_clean.withColumnRenamed(
            old_name, new_name
        )

# Check for duplicate column names after standardization
if len(df_clean.columns) != len(set(df_clean.columns)):
    raise ValueError("Duplicate column names after standardization.")

print("Selected source: aqhi.csv")
print("Standardized column names:")
print(df_clean.columns)

Standardized column names:
['date_pst', 'aqhi_area', 'station_name', 'station_name_full', 'latitude', 'longitude', 'cgndb', 'so2_source', 'no2_rounded', 'no2_rounded_3hr', 'o3_rounded', 'o3_rounded_3hr', 'pm25_rounded', 'pm25_rounded_3hr', 'so2_rounded', 'so2_rounded_3hr', 'active', 'aqhi_pm25', 'aqhi_so2', 'aqhi_classic', 'aqhi_plus_pm25', 'aqhi_value', 'so2_max', 'msg_index', 'aqhi_type', 'aqhi_reported', 'label_text', 'risk_pop_text', 'gen_pop_text', 'special_message', 'url', 'source', 'aqhi_plus_parameter', 'aqhi_plus_pm25_value']


### 5.1 Standardizing Missing Values

The source data may contain different representations of missing values, including empty strings, NA, N/A, NULL, and hyphens.

These representations are standardized to Spark NULL values to ensure consistent missing-value detection and processing.

In [ ]:
from pyspark.sql.types import StringType

# Identify string columns
string_columns = [
    field.name
    for field in df_clean.schema.fields
    if isinstance(field.dataType, StringType)
]

# Common missing-value representations
missing_values = ["", "NA", "N/A", "NULL", "-"]

# Standardize missing values
for column_name in string_columns:

    df_clean = df_clean.withColumn(
        column_name,
        F.when(
            F.upper(F.trim(F.col(column_name))).isin(
                [v.upper() for v in missing_values]
            ),
            None
        ).otherwise(F.trim(F.col(column_name)))
    )

print("Missing-value representations standardized.")

Standardized missing values.


### 5.2 Removing Exact Duplicate Records

In [ ]:
# Count records before duplicate removal
before_count = df_clean.count()

# Remove completely identical rows
df_clean = df_clean.dropDuplicates()

# Count records after duplicate removal
after_count = df_clean.count()

print("Records before cleaning:", before_count)
print("Records after cleaning:", after_count)
print("Exact duplicate records removed:", before_count - after_count)

Records before cleaning: 16494
Records after removing duplicates: 16494
Duplicates removed: 0


### 5.3 Validating Missing Values After Standardization

After standardizing missing-value representations and removing exact duplicate records, the dataset is checked again to identify NULL values across all columns.

In [ ]:
# Calculate missing values for every column
missing_counts = df_clean.select([
    F.count(
        F.when(F.col(column_name).isNull(), 1)
    ).alias(column_name)
    for column_name in df_clean.columns
])

missing_counts.show(truncate=False)

+--------+---------+------------+-----------------+--------+---------+-----+----------+-----------+---------------+----------+--------------+------------+----------------+-----------+---------------+------+---------+--------+------------+--------------+----------+-------+---------+---------+-------------+----------+-------------+------------+---------------+---+------+-------------------+--------------------+
|date_pst|aqhi_area|station_name|station_name_full|latitude|longitude|cgndb|so2_source|no2_rounded|no2_rounded_3hr|o3_rounded|o3_rounded_3hr|pm25_rounded|pm25_rounded_3hr|so2_rounded|so2_rounded_3hr|active|aqhi_pm25|aqhi_so2|aqhi_classic|aqhi_plus_pm25|aqhi_value|so2_max|msg_index|aqhi_type|aqhi_reported|label_text|risk_pop_text|gen_pop_text|special_message|url|source|aqhi_plus_parameter|aqhi_plus_pm25_value|
+--------+---------+------------+-----------------+--------+---------+-----+----------+-----------+---------------+----------+--------------+------------+----------------+---

### 5.4 Checking Data Types and Invalid Values

In [ ]:
df_clean.printSchema()

df_clean.show(10, truncate=False)

root
 |-- date_pst: timestamp (nullable = true)
 |-- aqhi_area: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- station_name_full: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- cgndb: string (nullable = true)
 |-- so2_source: string (nullable = true)
 |-- no2_rounded: double (nullable = true)
 |-- no2_rounded_3hr: double (nullable = true)
 |-- o3_rounded: double (nullable = true)
 |-- o3_rounded_3hr: double (nullable = true)
 |-- pm25_rounded: double (nullable = true)
 |-- pm25_rounded_3hr: double (nullable = true)
 |-- so2_rounded: double (nullable = true)
 |-- so2_rounded_3hr: double (nullable = true)
 |-- active: string (nullable = true)
 |-- aqhi_pm25: string (nullable = true)
 |-- aqhi_so2: string (nullable = true)
 |-- aqhi_classic: double (nullable = true)
 |-- aqhi_plus_pm25: integer (nullable = true)
 |-- aqhi_value: double (nullable = true)
 |-- so2_max: double (nullable = true)
 |-- ms

### 5.5 Investigating Special Missing Values

In [ ]:
# Identify numerical columns
numeric_columns = [
    field.name
    for field in df_clean.schema.fields
    if field.dataType.typeName() in [
        "double", "integer", "long", "float"
    ]
]

# Count special missing-value codes (-999)
for column in numeric_columns:

    count = df_clean.filter(
        F.col(column) == -999
    ).count()

    if count > 0:
        print(
            column,
            ":",
            count,
            "special missing values"
        )

aqhi_classic : 678 special missing values
aqhi_plus_pm25 : 168 special missing values
aqhi_value : 136 special missing values


### 5.6 Investigating Negative Pollutant Readings

In [ ]:
# Pollutant columns to investigate
pollutant_columns = [
    "no2_rounded",
    "no2_rounded_3hr",
    "o3_rounded",
    "o3_rounded_3hr",
    "pm25_rounded",
    "pm25_rounded_3hr",
    "so2_rounded",
    "so2_rounded_3hr"
]

# Count negative readings for each pollutant
for column in pollutant_columns:

    if column in df_clean.columns:

        count = df_clean.filter(
            F.col(column) < 0
        ).count()

        print(column, ":", count, "negative readings")

    else:
        print(column, ": column not found")

no2_rounded : 964 negative readings
no2_rounded_3hr : 929 negative readings
o3_rounded : 7 negative readings
o3_rounded_3hr : 1 negative readings
pm25_rounded : 0 negative readings
pm25_rounded_3hr : 0 negative readings
so2_rounded : 543 negative readings
so2_rounded_3hr : 549 negative readings


In [ ]:
df_clean.select(
    F.min("date_pst").alias("earliest_date"),
    F.max("date_pst").alias("latest_date"),
    F.countDistinct("date_pst").alias("unique_timestamps")
).show()

+-------------------+-------------------+-----------------+
|      earliest_date|        latest_date|unique_timestamps|
+-------------------+-------------------+-----------------+
|2026-10-06 01:00:00|2026-11-05 12:00:00|              732|
+-------------------+-------------------+-----------------+



### 5.7 Checking URL Field Consistency

In [ ]:
df_clean.filter(
    F.col("url").contains("|")
).select("station_name", "url").show(
    20, truncate=False
)

+------------+---+
|station_name|url|
+------------+---+
+------------+---+



### 5.8 Replacing Special Missing Values with NULL

In [ ]:
# Columns where -999 represents a special missing value
special_missing_columns = [
    "aqhi_classic",
    "aqhi_plus_pm25",
    "aqhi_value"
]

for column in special_missing_columns:

    if column in df_clean.columns:

        df_clean = df_clean.withColumn(
            column,
            F.when(
                F.col(column) == -999,
                None
            ).otherwise(F.col(column))
        )

print("Special missing values replaced with NULL.")

Special missing values replaced with NULL.


In [ ]:
# Verify that -999 values were successfully replaced
for column in special_missing_columns:

    if column in df_clean.columns:

        count = df_clean.filter(
            F.col(column) == -999
        ).count()

        print(
            column,
            "remaining -999 values:",
            count
        )

aqhi_classic remaining -999 values: 0
aqhi_plus_pm25 remaining -999 values: 0
aqhi_value remaining -999 values: 0


### 5.9 Investigate Negative pollutant readings

In [ ]:
# Calculate minimum and maximum values for each pollutant
df_clean.select([
    F.min(c).alias(c + "_minimum")
    for c in pollutant_columns
] + [
    F.max(c).alias(c + "_maximum")
    for c in pollutant_columns
]).show(truncate=False)

+-------------------+-----------------------+------------------+----------------------+--------------------+------------------------+-------------------+-----------------------+-------------------+-----------------------+------------------+----------------------+--------------------+------------------------+-------------------+-----------------------+
|no2_rounded_minimum|no2_rounded_3hr_minimum|o3_rounded_minimum|o3_rounded_3hr_minimum|pm25_rounded_minimum|pm25_rounded_3hr_minimum|so2_rounded_minimum|so2_rounded_3hr_minimum|no2_rounded_maximum|no2_rounded_3hr_maximum|o3_rounded_maximum|o3_rounded_3hr_maximum|pm25_rounded_maximum|pm25_rounded_3hr_maximum|so2_rounded_maximum|so2_rounded_3hr_maximum|
+-------------------+-----------------------+------------------+----------------------+--------------------+------------------------+-------------------+-----------------------+-------------------+-----------------------+------------------+----------------------+--------------------+--------

In [ ]:
# Count negative pollutant readings for each monitoring station
df_clean.groupBy("station_name").agg(
    F.sum(
        F.when(F.col("no2_rounded") < 0, 1).otherwise(0)
    ).alias("negative_no2"),

    F.sum(
        F.when(F.col("so2_rounded") < 0, 1).otherwise(0)
    ).alias("negative_so2"),

    F.sum(
        F.when(F.col("o3_rounded") < 0, 1).otherwise(0)
    ).alias("negative_o3")
).orderBy(
    F.desc("negative_no2")
).show(20, truncate=False)

+--------------------------------+------------+------------+-----------+
|station_name                    |negative_no2|negative_so2|negative_o3|
+--------------------------------+------------+------------+-----------+
|Terrace Skeena Middle School    |338         |1           |0          |
|Quesnel Johnston Avenue         |306         |166         |0          |
|Williams Lake Columneetza School|184         |0           |2          |
|Smithers Muheim Memorial        |54          |0           |0          |
|Colwood City Hall               |37          |0           |0          |
|Fort St John Key Learning Centre|36          |0           |0          |
|Prince George Plaza 400         |8           |0           |1          |
|Castlegar Zinio Park            |1           |0           |1          |
|Squamish Elementary             |0           |11          |0          |
|Kamloops Federal Building       |0           |0           |0          |
|Sparwood Centennial Square      |0           |0   

In [ ]:
import builtins

# Calculate total number of records once
total_records = df_clean.count()

# Calculate negative readings and their percentages
for column in pollutant_columns:

    negative = df_clean.filter(
        F.col(column) < 0
    ).count()

    percentage = (negative / total_records) * 100

    print(
        column,
        ":",
        negative,
        "records (",
        builtins.round(percentage, 2),
        "%)"
    )

no2_rounded : 964 records ( 5.84 %)
no2_rounded_3hr : 929 records ( 5.63 %)
o3_rounded : 7 records ( 0.04 %)
o3_rounded_3hr : 1 records ( 0.01 %)
pm25_rounded : 0 records ( 0.0 %)
pm25_rounded_3hr : 0 records ( 0.0 %)
so2_rounded : 543 records ( 3.29 %)
so2_rounded_3hr : 549 records ( 3.33 %)


### 5.10 Creating Data-Quality Flags

Negative pollutant measurements are flagged rather than automatically removed.

A separate binary flag is created for each pollutant, where 1 indicates a negative reading and 0 indicates a non-negative reading.

An additional overall flag identifies records containing at least one negative pollutant measurement.

These flags allow potentially problematic observations to be identified and investigated during subsequent analysis.

In [ ]:
import builtins

# Create a flag for each pollutant's negative readings
for column in pollutant_columns:

    flag_name = column + "_negative_flag"

    df_clean = df_clean.withColumn(
        flag_name,
        F.when(F.col(column) < 0, 1).otherwise(0)
    )

# Collect all individual negative-reading flags
flag_cols = [
    F.col(column + "_negative_flag")
    for column in pollutant_columns
]

# Calculate the total number of negative pollutant readings per record
total_negatives = builtins.sum(
    flag_cols,
    F.lit(0)
)

# Create an overall data-quality flag
df_clean = df_clean.withColumn(
    "has_negative_reading",
    F.when(total_negatives > 0, 1).otherwise(0)
)

print("Individual and overall data-quality flags created.")

Data-quality flags created.


### 5.11 Calculating Missing Values Across Columns

In [ ]:
total_records = df_clean.count()

missing_summary = df_clean.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in df_clean.columns
])

missing_summary.show(truncate=False)

+--------+---------+------------+-----------------+--------+---------+-----+----------+-----------+---------------+----------+--------------+------------+----------------+-----------+---------------+------+---------+--------+------------+--------------+----------+-------+---------+---------+-------------+----------+-------------+------------+---------------+---+------+-------------------+--------------------+-------------------------+-----------------------------+------------------------+----------------------------+--------------------------+------------------------------+-------------------------+-----------------------------+--------------------+
|date_pst|aqhi_area|station_name|station_name_full|latitude|longitude|cgndb|so2_source|no2_rounded|no2_rounded_3hr|o3_rounded|o3_rounded_3hr|pm25_rounded|pm25_rounded_3hr|so2_rounded|so2_rounded_3hr|active|aqhi_pm25|aqhi_so2|aqhi_classic|aqhi_plus_pm25|aqhi_value|so2_max|msg_index|aqhi_type|aqhi_reported|label_text|risk_pop_text|gen_pop_tex

In [ ]:
import builtins

missing_report = []

for column in df_clean.columns:
    missing_count = df_clean.filter(
        F.col(column).isNull()
    ).count()

    missing_percentage = (
        (missing_count / total_records) * 100
        if total_records > 0 else 0
    )

    missing_report.append(
        (
            column,
            missing_count,
            builtins.round(missing_percentage, 2)
        )
    )

missing_df = spark.createDataFrame(
    missing_report,
    ["Column", "Missing_Count", "Missing_Percentage"]
)

missing_df.orderBy(
    F.desc("Missing_Percentage")
).show(40, truncate=False)

+--------------------+-------------+------------------+
|Column              |Missing_Count|Missing_Percentage|
+--------------------+-------------+------------------+
|SPECIAL_MESSAGE     |16494        |100.0             |
|SO2_SOURCE          |15762        |95.56             |
|SO2_MAX             |15762        |95.56             |
|SOURCE              |15372        |93.2              |
|AQHI_PLUS_PARAMETER |15372        |93.2              |
|AQHI_PLUS_PM25_VALUE|15372        |93.2              |
|SO2_ROUNDED         |9563         |57.98             |
|SO2_ROUNDED_3HR     |9312         |56.46             |
|O3_ROUNDED          |2078         |12.6              |
|NO2_ROUNDED         |1954         |11.85             |
|AQHI_CLASSIC        |1800         |10.91             |
|O3_ROUNDED_3HR      |1526         |9.25              |
|NO2_ROUNDED_3HR     |1384         |8.39              |
|PM25_ROUNDED_3HR    |1291         |7.83              |
|AQHI_PLUS_PM25      |1290         |7.82        

In [ ]:
df_clean.filter(
    F.col("station_name").isNull()
).select(
    "date_pst",
    "aqhi_area",
    "aqhi_value",
    "aqhi_classic",
    "aqhi_type",
    "aqhi_reported"
).show(15, truncate=False)

+-------------------+---------------------+----------+------------+---------+-------------+
|date_pst           |aqhi_area            |aqhi_value|aqhi_classic|aqhi_type|aqhi_reported|
+-------------------+---------------------+----------+------------+---------+-------------+
|2026-11-02 01:00:00|Metro Vancouver NW   |1.36      |NULL        |CLASSIC  |1            |
|2026-11-04 03:00:00|Central Fraser Valley|1.37      |NULL        |CLASSIC  |1            |
|2026-10-31 06:00:00|Metro Vancouver SE   |1.33      |NULL        |CLASSIC  |1            |
|2026-10-29 02:00:00|Eastern Fraser Valley|1.14      |NULL        |CLASSIC  |1            |
|2026-10-29 03:00:00|Eastern Fraser Valley|1.15      |NULL        |CLASSIC  |1            |
|2026-10-31 17:00:00|Metro Vancouver SW   |1.75      |NULL        |CLASSIC  |2            |
|2026-11-01 00:00:00|Metro Vancouver SW   |1.83      |NULL        |CLASSIC  |2            |
|2026-11-03 09:00:00|Metro Vancouver SW   |1.74      |NULL        |CLASSIC  |2  

In [ ]:
df_clean.select(
    "aqhi_plus_pm25",
    "aqhi_plus_pm25_value",
    "aqhi_plus_parameter",
    "aqhi_value"
).show(20, truncate=False)

+--------------+--------------------+-------------------+----------+
|aqhi_plus_pm25|aqhi_plus_pm25_value|aqhi_plus_parameter|aqhi_value|
+--------------+--------------------+-------------------+----------+
|1             |NULL                |NULL               |1.0       |
|1             |NULL                |NULL               |1.76      |
|3             |NULL                |NULL               |3.0       |
|1             |NULL                |NULL               |1.0       |
|2             |NULL                |NULL               |2.0       |
|2             |NULL                |NULL               |2.0       |
|2             |NULL                |NULL               |2.0       |
|1             |NULL                |NULL               |1.82      |
|1             |NULL                |NULL               |2.11      |
|1             |NULL                |NULL               |1.0       |
|2             |NULL                |NULL               |2.0       |
|1             |NULL              

In [ ]:
columns_to_remove = [
    "special_message",
    "so2_source",
    "so2_max",
    "source",
    "aqhi_plus_parameter"
]

existing_columns_to_remove = [
    column for column in columns_to_remove
    if column in df_clean.columns
]

df_clean = df_clean.drop(*existing_columns_to_remove)

print("Removed columns:", existing_columns_to_remove)
print("Remaining columns:", df_clean.columns)

Removed unnecessary columns:
['special_message', 'so2_source', 'so2_max', 'source', 'aqhi_plus_parameter']
Remaining columns: 38


In [ ]:
duplicate_station_records = (
    df_clean
    .groupBy("station_name", "date_pst")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate station-timestamp combinations:",
    duplicate_station_records.count()
)

duplicate_station_records.show(20, truncate=False)

Duplicate station-timestamp combinations: 187
+------------+-------------------+-----+
|station_name|date_pst           |count|
+------------+-------------------+-----+
|NULL        |2026-11-01 15:00:00|6    |
|NULL        |2026-10-30 12:00:00|6    |
|NULL        |2026-11-03 09:00:00|6    |
|NULL        |2026-10-31 17:00:00|6    |
|NULL        |2026-11-05 10:00:00|6    |
|NULL        |2026-10-29 02:00:00|6    |
|NULL        |2026-11-01 13:00:00|6    |
|NULL        |2026-11-01 01:00:00|6    |
|NULL        |2026-11-04 19:00:00|6    |
|NULL        |2026-11-02 18:00:00|6    |
|NULL        |2026-11-05 04:00:00|6    |
|NULL        |2026-10-31 10:00:00|6    |
|NULL        |2026-11-04 12:00:00|6    |
|NULL        |2026-11-01 19:00:00|6    |
|NULL        |2026-11-03 18:00:00|6    |
|NULL        |2026-11-02 16:00:00|6    |
|NULL        |2026-10-29 15:00:00|6    |
|NULL        |2026-11-05 11:00:00|6    |
|NULL        |2026-10-31 04:00:00|6    |
|NULL        |2026-11-01 10:00:00|6    |
+----------

In [ ]:
# Check duplicates using station, AQHI area and timestamp

duplicate_check = (
    df_clean
    .groupBy("station_name", "aqhi_area", "date_pst")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_check.count()

print(
    "Duplicate combinations using station, AQHI area and timestamp:",
    duplicate_count
)

duplicate_check.show(30, truncate=False)

Duplicate combinations after including AQHI area: 0
+------------+---------+--------+-----+
|station_name|aqhi_area|date_pst|count|
+------------+---------+--------+-----+
+------------+---------+--------+-----+



In [ ]:
# Check duplicate records for monitoring stations only

station_duplicates = (
    df_clean
    .filter(F.col("station_name").isNotNull())
    .groupBy("station_name", "date_pst")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate non-null monitoring-station records:",
    station_duplicates.count()
)

station_duplicates.show(20, truncate=False)

Duplicate non-null station records: 0
+------------+--------+-----+
|station_name|date_pst|count|
+------------+--------+-----+
+------------+--------+-----+



In [ ]:
# Check the distribution of regional AQHI records

regional_record_counts = (
    df_clean
    .filter(F.col("station_name").isNull())
    .groupBy("aqhi_area")
    .count()
    .orderBy(F.desc("count"))
)

print("Regional AQHI records by area:")
regional_record_counts.show(30, truncate=False)

+---------------------+-----+
|aqhi_area            |count|
+---------------------+-----+
|Metro Vancouver NW   |187  |
|Metro Vancouver NE   |187  |
|Eastern Fraser Valley|187  |
|Metro Vancouver SE   |187  |
|Metro Vancouver SW   |187  |
|Central Fraser Valley|187  |
+---------------------+-----+



In [ ]:
# Check availability of AQHI Plus PM2.5 values

aqhi_plus_summary = df_clean.select(
    F.count("aqhi_plus_pm25_value").alias("Available_Values"),
    F.count("*").alias("Total_Records")
)

aqhi_plus_summary.show(truncate=False)

+----------------+-------------+
|Available_values|Total_records|
+----------------+-------------+
|            1122|        16494|
+----------------+-------------+



## 6. Separating Monitoring Station and Regional AQHI Datasets

In [ ]:
# Dataset A: Individual monitoring station observations

station_df = df_clean.filter(
    F.col("station_name").isNotNull()
)

# Dataset B: Regional AQHI observations

regional_df = df_clean.filter(
    F.col("station_name").isNull()
)

# Calculate record counts
station_count = station_df.count()
regional_count = regional_df.count()
original_count = df_clean.count()

print("Monitoring Station Dataset:")
print("Records:", station_count)

print("\nRegional AQHI Dataset:")
print("Records:", regional_count)

print("\nOriginal Cleaned Dataset:")
print("Records:", original_count)

print("\nCombined Records:")
print(station_count + regional_count)

# Verify that all records are accounted for
if station_count + regional_count == original_count:
    print("\nValidation successful: No records were lost.")
else:
    raise ValueError("Dataset separation validation failed.")

Monitoring Station Dataset:
Records: 15372

Regional AQHI Dataset:
Records: 1122

Combined Records:
16494


In [ ]:
# Validate Regional AQHI Dataset

print("Regional AQHI Dataset Validation:")

regional_df.select(
    F.count("*").alias("Total_Records"),
    F.count("aqhi_value").alias("Available_AQHI"),
    F.count("aqhi_plus_pm25_value").alias(
        "Available_AQHI_Plus_PM25"
    )
).show(truncate=False)


# Validate Monitoring Station Dataset

print("Monitoring Station Dataset Validation:")

station_df.select(
    F.count("*").alias("Total_Records"),
    F.count("no2_rounded").alias("Available_NO2"),
    F.count("o3_rounded").alias("Available_O3"),
    F.count("pm25_rounded").alias("Available_PM25"),
    F.count("so2_rounded").alias("Available_SO2")
).show(truncate=False)


# Confirm dataset separation

print("Monitoring station records:", station_df.count())
print("Regional AQHI records:", regional_df.count())

Regional AQHI Dataset Check:
+-------------+--------------+------------------------+
|Total_Records|Available_AQHI|Available_AQHI_Plus_PM25|
+-------------+--------------+------------------------+
|         1122|          1122|                    1122|
+-------------+--------------+------------------------+

Monitoring Station Dataset Check:
+-------------+-------------+------------+--------------+-------------+
|Total_Records|Available_NO2|Available_O3|Available_PM25|Available_SO2|
+-------------+-------------+------------+--------------+-------------+
|        15372|        14540|       14416|         15204|         6931|
+-------------+-------------+------------+--------------+-------------+



## 7. Saving the Cleaned Datasets in Parquet Format

In [ ]:
# Define output paths
station_output_path = PROCESSED_DIR / "station_data"
regional_output_path = PROCESSED_DIR / "regional_aqhi_data"

# Save monitoring station dataset
station_df.write \
    .mode("overwrite") \
    .parquet(str(station_output_path))

# Save regional AQHI dataset
regional_df.write \
    .mode("overwrite") \
    .parquet(str(regional_output_path))

print("Both datasets saved successfully!")

print("Monitoring station data:", station_output_path)
print("Regional AQHI data:", regional_output_path)

Both cleaned datasets saved successfully!
